In [1]:
from qiskit import QuantumCircuit
import qiskit
from qiskit import transpile
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Operator
from math import pi
import numpy as np
from matplotlib import pyplot as plt

from sqlalchemy.orm import joinedload
from sqlalchemy import select, func

from benchmarklib import BenchmarkDatabase
from benchmarklib.runners import BatchQueue

from experiments import ExperimentProblem, ExperimentTrial

TAG = "toffoli_layout_variation"

service = QiskitRuntimeService()
backend = service.backend("ibm_rensselaer")

db = BenchmarkDatabase("experiments.db", ExperimentProblem, ExperimentTrial)

In [3]:
def relative_phase_toffoli_layout_default():
    qc = QuantumCircuit(3, 3)
    qc.rccx(0, 1, 2)
    return qc

def toffoli_18():
    qc = QuantumCircuit(3)
    qc.swap(1, 2)

    qc.rz(-pi/2, 0)
    qc.sx(1)
    qc.rz(pi/2, 1)
    qc.rz(-pi/2, 2)
    qc.ecr(0, 1)

    qc.x(0)
    qc.rz(-pi/2, 0)
    qc.rz(3*pi/4, 1)
    qc.sx(1)
    qc.rz(-pi,1)
    qc.ecr(2, 1)

    qc.rz(-3*pi/4,1)
    qc.sx(1)
    qc.rz(-pi, 1)
    qc.x(2)
    qc.rz(-pi/2,2)
    qc.ecr(0,1)

    qc.rz(-3*pi/4,0)
    qc.sx(0)
    qc.rz(3*pi/4,1)
    qc.sx(1)
    qc.rz(-pi,1)
    qc.ecr(2,1)

    qc.rz(-3*pi/4,1)
    qc.sx(1)
    qc.rz(pi/2,1)
    qc.rz(-pi,2)
    qc.x(2)
    qc.ecr(2,1)

    qc.sx(1)
    qc.rz(-pi/2,1)
    qc.rz(-pi/2,2)
    qc.sx(2)
    qc.ecr(2,1)

    qc.rz(-pi,1)
    qc.sx(1)
    qc.rz(pi/2,1)
    qc.rz(pi/2,2)
    qc.sx(2)
    qc.ecr(2,1)
    
    qc.rz(pi/2,1)
    qc.sx(1)
    qc.rz(-pi/2,1)
    qc.ecr(0,1)

    qc.rz(-pi/2,0)
    qc.sx(0)
    qc.rz(pi/4,0)
    qc.sx(0)
    qc.rz(pi/2,1)
    qc.sx(1)
    qc.rz(-3*pi/4,1)
    qc.sx(1)
    qc.rz(pi/2,1)
    qc.ecr(0,1)

    qc.rz(pi/2,0)
    qc.sx(0)
    qc.rz(pi/2,0)
    qc.rz(-pi/2,1)
    qc.sx(1)
    qc.rz(pi/2,1)

    return qc

In [41]:
def create_trials(toffoli_circuit, name, coupling_map):
    """
    This experiment compares the toffoli performance based on where it is 
    placed within the connectivity graph. Use coupling_map to specify which qubits
    the toffoli should be transpiled to use. Note: we do not care which layout is used
    within that coupling_map, and rather assume the transpiler is good at finding "the best" one within it.
    """
    trials = []
    # check if problem already existis
    existing = db.query(select(ExperimentProblem).where(ExperimentProblem.name == name, ExperimentProblem.tag == TAG))
    if len(existing) > 0:
        experiment = existing[0]
    else:
        experiment = ExperimentProblem(
            name=name,
            tag=TAG,
            n=3,
        )

    for i in range(2**3):
        input_state = format(i, '03b')
        qc = QuantumCircuit(3, 3)
        for qubit in range(3):
            if input_state[qubit] == '1':
                qc.x(qubit)

                
        qc.compose(toffoli_circuit, inplace=True)
        qc.measure(range(3), range(3))

        qc_final = transpile(
            qc,
            backend=backend,
            coupling_map=coupling_map,
            optimization_level=3,
            routing_method='sabre',
            basis_gates=None,
        )
  
        trials.append(
            ExperimentTrial(problem=experiment, circuit=qc_final, circuit_pretranspile=qc, extra_data={"input_state": input_state, "backend": backend.name})
        )

    return trials

    

In [42]:
from qiskit.transpiler import CouplingMap
coupling_map = backend.coupling_map

def get_coupling_map_around_qubit(qubit):
    """
    Returns a coupling map that only includes the specified qubit and its neighbors.
    """
    edges = []
    for edge in coupling_map.get_edges():
        if edge[0] == qubit or edge[1] == qubit:
            edges.append(edge)
    return CouplingMap(edges)

In [47]:
trials = create_trials(toffoli_18(), f"toffoli_18+{0}", coupling_map=get_coupling_map_around_qubit(10))

/home/eriku/miniconda3/envs/tweedledum-dev/lib/python3.10/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


In [56]:
with BatchQueue(db, backend=backend, shots=4096) as q:
    for circuit, name in [
        (toffoli_18(), "toffoli_18"),
        (relative_phase_toffoli_layout_default(), "relative_phase_toffoli_layout_default"),
    ]:
        for qubit in range(127):
            if qubit in [13, 113]:
                # these qubits cannot be centers as they only are connected to one other qubit
                continue
            for trial in create_trials(circuit, f"{name}+{qubit}", coupling_map=get_coupling_map_around_qubit(qubit)):
                q.enqueue(trial, trial.circuit, run_simulation=True)
    q.submit_tasks()

/home/eriku/miniconda3/envs/tweedledum-dev/lib/python3.10/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


In [57]:
await db.update_all_pending_results(service=service)

### Compare Regions of Connectivity Graph / Coupling Map

In [3]:
from typing import Optional
def properties_aggregate(properties_dict : dict, qubit_indices: Optional[set] = None) -> dict:
    """
    Converts the list of dictionaries into a dictionary of lists for each of "gates" and "qubits" 
    within properties_dict. 
    If qubit_indices is specified, this ignors any properties that include a qubit not in qubit_indices.
    """
    qubits = {}
    gates = {}
    for i, q_props in enumerate(properties_dict["qubits"]):
        if qubit_indices is not None and i not in qubit_indices:
            # skip over properties for qubits we are not considering
            continue
        for prop in q_props:
            if prop["name"] not in qubits:
                qubits[prop["name"]] = []
            qubits[prop["name"]].append(prop["value"])
    for g_props in properties_dict["gates"]:
        if qubit_indices is not None and any(q not in qubit_indices for q in g_props["qubits"]):
            # skip over properties for gates that include a qubits we are not considering
            continue
        if g_props["gate"] not in gates:
            gates[g_props["gate"]] = {}
        for prop in g_props["parameters"]:
            if prop["name"] not in gates[g_props["gate"]]:
                gates[g_props["gate"]][prop["name"]] = []
            gates[g_props["gate"]][prop["name"]].append(prop["value"])
    return {
        "qubits": qubits,
        "gates": gates,
    }

def properties_aggregate_metrics(properties_dict: dict, qubit_indices: Optional[set] = None) -> dict:
    """
    Calculates aggregate metrics (mean, std, min, max, median) for each qubit and each gate type in properties_dict.
    If qubit_indices is specified, this ignors any properties that include a qubit not in qubit_indices.
    """
    qubits = {}
    gates = {}
    p = properties_aggregate(properties_dict, qubit_indices)
    for key, values in p["qubits"].items():
        qubits[key] = {}
        qubits[key]["mean"] = np.mean(values)
        qubits[key]["std"] = np.std(values)
        qubits[key]["min"] = np.min(values)
        qubits[key]["max"] = np.max(values)
        qubits[key]["median"] = np.median(values)
    for gate_type, props in p["gates"].items():
        for key, values in props.items():
            if gate_type not in gates:
                gates[gate_type] = {}
            gates[gate_type][key] = {}
            gates[gate_type][key]["mean"] = np.mean(values)
            gates[gate_type][key]["std"] = np.std(values)
            gates[gate_type][key]["min"] = np.min(values)
            gates[gate_type][key]["max"] = np.max(values)
            gates[gate_type][key]["median"] = np.median(values)
    return {
        "qubits": qubits,
        "gates": gates,
    }

def flatten_dict(d: dict) -> dict:
    """
    Given nested dict d, returns dict flattened to one layer deep using key concatination.
    Expects all keys to be strings, all values from a layer to either be dictionaries (potentially nested)
    or flat values (no mixing of the two at a layer).
    This utility is usefully applied to the output of properties_aggregate_metrics if a flat dictionary
    is desired.
    """
    # check if already flat
    if not isinstance(list(d.values())[0], dict):
        return d
    
    res = {}
    for key, child in d.items():
        for child_key, child_value in flatten_dict(child).items():
            res[key + "_" + child_key] = child_value

    return res


In [ ]:
d1 = flatten_dict(properties_aggregate_metrics(backend.properties().to_dict()))
d2 = flatten_dict(properties_aggregate_metrics(backend.properties().to_dict(), set(range(93, 127))))
d3 = flatten_dict(properties_aggregate_metrics(backend.properties().to_dict(), set(range(56, 76))))

for key in d1.keys():
    print(f"{key}: {d1[key]:.4f} | {d2[key]:.4f} | {d3[key]:.4f}")

### Visualization

In [60]:
results = db.query(
        select(ExperimentTrial)
        .join(ExperimentProblem)
        .where(ExperimentProblem.name.like(f"relative_phase_toffoli_layout_default+%"), ExperimentProblem.tag == TAG)
        .options(joinedload(ExperimentTrial.problem))
    )

In [ ]:
from qiskit.visualization import plot_coupling_map, plot_gate_map
import matplotlib.pyplot as plt
import matplotlib as mpl

def visualize_layout_results(toffoli_name, flip_vertical=False):
    results = db.query(
        select(ExperimentTrial)
        .join(ExperimentProblem)
        .where(ExperimentProblem.name.like(f"{toffoli_name}+%"), ExperimentProblem.tag == TAG)
        .options(joinedload(ExperimentTrial.problem))
    )
    
    # first, group fidelity results by the qubit index used as the center
    qubit_center_results = {}
    for trial in results:
        qubit = int(trial.problem.name.split("+")[1])
        if qubit not in qubit_center_results:
            qubit_center_results[qubit] = []
        qubit_center_results[qubit].append(trial.calculate_success_rate())
    
    
    # create a list of average fidelity per center qubit to use for coloring in the plot
    ordered_results = []
    for idx in range(127):
        if idx not in qubit_center_results:
            ordered_results.append(0.0)
            continue
        ordered_results.append(np.mean(qubit_center_results[idx]))
    
    # convert fidelity averages into colors using a color map
    norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
    cmap = plt.get_cmap("viridis")
    colors = cmap(norm(ordered_results))
    rgb_strings = [
        f"#{hex(int(r*255))[2:].zfill(2)}{hex(int(g*255))[2:].zfill(2)}{hex(int(b*255))[2:].zfill(2)}"
        for r, g, b, _ in colors
    ]

    fig = plot_gate_map(backend, qubit_color=rgb_strings, qubit_size=60, font_color="black")
    if flip_vertical:
        # flip to match orientation of the IBM coupling map plot
        fig.gca().invert_yaxis()

    # add colorbar to figure
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=fig.axes[0])
    cbar.set_label("Average Fidelity")

    return fig


In [ ]:
visualize_layout_results("toffoli_18")